In [1]:
import torch
import sys
import time

sys.path.append("/home/atuin/v120bb/v120bb18/UnReflectAnything")
from utilities.visualization import rgb, panelize
from polar_highlighter import PolarHighlighter

if torch.cuda.is_available():
    num_devices = torch.cuda.device_count()
    curr_device = torch.cuda.current_device()
    device_name = torch.cuda.get_device_name(curr_device)
    print(f"CUDA is available: {num_devices} device(s) detected.")
    print(f"Current device id: {curr_device} - {device_name}")
else:
    print("CUDA is not available")
%load_ext autoreload
%autoreload 2


CUDA is available: 1 device(s) detected.
Current device id: 0 - NVIDIA A40


In [2]:
MODELS = ["icy-gorge-754","easy-surf-759"]

In [ ]:
from main import load_and_process_config
from models_utils import load_best_model_by_run
from dataset import from_config
from utilities import tensor_dict_summarize


config = load_and_process_config("config_train.yaml")
config.RUN = "easy-surf-759"
# config.DATASETS = {"PSD": config.DATASETS.PSD}

dataset = from_config(config)["validation"]
idataloadr = iter(torch.utils.data.DataLoader(dataset, batch_size=1, shuffle=False))
model = load_best_model_by_run(config.RUN).eval()

In [ ]:
import importlib
import models
importlib.reload(models)
config = load_and_process_config("config_train.yaml")
config.RUN = "easy-surf-759"
model = load_best_model_by_run(config.RUN).eval()
dataloader = torch.utils.data.DataLoader(dataset, batch_size=1, shuffle=True)

for b, batch in enumerate(dataloader):
    batch = {
        k: v.cuda() if isinstance(v, torch.Tensor) and torch.cuda.is_available() else v
        for k, v in batch.items()
    }
    patch_mask = batch["raw"].mean(dim=1, keepdim=True) > 0.9*batch["raw"].mean(dim=1).max().item()
    # model_input = {"rgb": batch["raw"], "inpaint_mask_override": patch_mask}
    model_input = {"rgb": batch["raw"]}
    
    with torch.no_grad():
        modelout = model(model_input)
    rgb(batch["raw"][0], as_tensor=False, resize=(448, 448)),
    rgb(modelout["diffuse"][0], as_tensor=False, resize=(448, 448)),
    # rgb(
    #     panelize(
    #         rgb(batch["raw"][0], as_tensor=True, resize=(448, 448)),
    #         rgb(modelout["diffuse"][0], as_tensor=True, resize=(448, 448)),
    #         rgb(modelout["highlight"][0], as_tensor=True, resize=(448, 448), colormap="gray"),
    #         # rgb(patch_mask.int()[0], as_tensor=True, resize=(448, 448), colormap="gray"),
    #     )
    # )
    if b > 20:
        break
    


In [ ]:
    _, pca = rgb(
        modelout["tokens_completed"][-1].reshape(1, 28, 28, 1024).permute(0, 3, 1, 2),
        as_tensor=True,
        resize=(448, 448),
        blackout=True,
        return_pca=True,
    )
    
    rgb(
        panelize(
            rgb(batch["raw"][0], as_tensor=True, resize=(448, 448)),
            # rgb(
            #     modelout["tokens_completed"][-1]
            #     .reshape(1, 28, 28, 1024)
            #     .permute(0, 3, 1, 2),
            #     as_tensor=True,
            #     pca=pca,
            #     resize=(448, 448),
            # ),
            # rgb(
            #     modelout["tokens_completed"][-1]
            #     .reshape(1, 28, 28, 1024)
            #     .permute(0, 3, 1, 2)
            #     * torch.logical_not(modelout["patch_mask"].reshape(1, 1, 28, 28)),
            #     as_tensor=True,
            #     resize=(448, 448),
            #     pca=pca,
            #     blackout=True,
            # ),
            rgb(modelout["diffuse"][0], as_tensor=True, resize=(448, 448)),
            rgb(modelout["highlight"][0], as_tensor=True, resize=(448, 448)),
        )
    )
    if b > 3:
        break


In [5]:
config = load_and_process_config("config_train.yaml")
config.RUN = "curious-moon-751"
config.DATASETS = {"PSD": config.DATASETS.PSD}
dataset = from_config(config)["validation"]
idataloadr = iter(torch.utils.data.DataLoader(dataset, batch_size=1, shuffle=False))
model = load_best_model_by_run(config.RUN).eval()
model.token_inpaint._prior_kernel = 5


In [6]:
print(model.token_inpaint._prior_kernel)

In [ ]:
dataloader = torch.utils.data.DataLoader(dataset, batch_size=1, shuffle=False)

for b, batch in enumerate(dataloader):
    batch = {
        k: v.cuda() if isinstance(v, torch.Tensor) and torch.cuda.is_available() else v
        for k, v in batch.items()
    }
    
    patch_mask = batch["raw"].mean(dim=1, keepdim=True) > 0.8
    model_input = {"rgb": batch["raw"]} #, "inpaint_mask_override": patch_mask}
    
    with torch.no_grad():
        modelout = model(model_input)
    _, pca = rgb(
        modelout["tokens_completed"][-1].reshape(1, 28, 28, 1024).permute(0, 3, 1, 2),
        as_tensor=True,
        resize=(448, 448),
        blackout=True,
        return_pca=True,
    )
    rgb(
        panelize(
            rgb(batch["raw"][0], as_tensor=True, resize=(448, 448)),
            # rgb(
            #     modelout["tokens_completed"][-1]
            #     .reshape(1, 28, 28, 1024)
            #     .permute(0, 3, 1, 2),
            #     as_tensor=True,
            #     pca=pca,
            #     resize=(448, 448),
            # ),
            # rgb(
            #     modelout["tokens_completed"][-1]
            #     .reshape(1, 28, 28, 1024)
            #     .permute(0, 3, 1, 2)
            #     * torch.logical_not(modelout["patch_mask"].reshape(1, 1, 28, 28)),
            #     as_tensor=True,
            #     resize=(448, 448),
            #     pca=pca,
            #     blackout=True,
            # ),
            rgb(modelout["diffuse"][0], as_tensor=True, resize=(448, 448)),
            rgb(modelout["highlight"][0], as_tensor=True, resize=(448, 448)),
        )
    )
    if b > 3:
        break


In [8]:
config = load_and_process_config("config_train.yaml")
config.RUN = "curious-moon-751"

# config.DATASETS = {"PSD": config.DATASETS.PSD}
# dataset = from_config(config)["validation"]
# idataloadr = iter(torch.utils.data.DataLoader(dataset, batch_size=1, shuffle=False))
model = load_best_model_by_run(config.RUN).eval()
# model.token_inpaint._prior_kernel = 11

dataloader = torch.utils.data.DataLoader(dataset, batch_size=1, shuffle=False)

for b, batch in enumerate(dataloader):
    batch = {
        k: v.cuda() if isinstance(v, torch.Tensor) and torch.cuda.is_available() else v
        for k, v in batch.items()
    }
    
    patch_mask = batch["raw"].mean(dim=1, keepdim=True) > 0.8
    model_input = {"rgb": batch["raw"]} #, "inpaint_mask_override": patch_mask}
    
    with torch.no_grad():
        modelout = model(model_input)
    _, pca = rgb(
        modelout["tokens_completed"][-1].reshape(1, 28, 28, 1024).permute(0, 3, 1, 2),
        as_tensor=True,
        resize=(448, 448),
        blackout=True,
        return_pca=True,
    )
    rgb(
        panelize(
            rgb(batch["raw"][0], as_tensor=True, resize=(448, 448)),
            # rgb(
            #     modelout["tokens_completed"][-1]
            #     .reshape(1, 28, 28, 1024)
            #     .permute(0, 3, 1, 2),
            #     as_tensor=True,
            #     pca=pca,
            #     resize=(448, 448),
            # ),
            # rgb(
            #     modelout["tokens_completed"][-1]
            #     .reshape(1, 28, 28, 1024)
            #     .permute(0, 3, 1, 2)
            #     * torch.logical_not(modelout["patch_mask"].reshape(1, 1, 28, 28)),
            #     as_tensor=True,
            #     resize=(448, 448),
            #     pca=pca,
            #     blackout=True,
            # ),
            rgb(modelout["diffuse"][0], as_tensor=True, resize=(448, 448)),
            rgb(modelout["highlight"][0], as_tensor=True, resize=(448, 448)),
        )
    )
    if b > 3:
        break


In [ ]:
# torch.save(model.token_inpaint.state_dict(), "/home/atuin/v120bb/v120bb18/UnReflectAnything/weights/token_inpainter.pth")